# 🩺 Diabetic Retinopathy Grading — Production Pipeline v21
## APTOS 2019 Blindness Detection | 1st-Place Strategy Replication
---

**Strategy:** Replicates the verified 1st-place Kaggle solution:
- **Minimal preprocessing** (resize only — no CLAHE, no Ben Graham)
- **SmoothL1Loss** regression (single output neuron)
- **GeM pooling** (learnable)
- **8-model ensemble** (4 backbones × 2 seeds)
- **Pseudo-labeling** (Stage 2)
- **Threshold tuning** → final `[0.7, 1.5, 2.5, 3.5]`

**Targets:** QWK ≈ 0.93–0.936 | Accuracy ≥ 95% | F1 ≥ 95%

**Hardware:** NVIDIA RTX 2050 (CUDA) / Apple MPS / CPU — auto-detected

**Resume:** Every step saves state. Crash → Re-run → Resumes exactly.

⚠️ **RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT**

In [1]:
import os, sys
from pathlib import Path

# Force R: drive for this session
os.environ["DR_BASE"] = "R:\\DR_Grading_v21"

# Verify it's set
print(f"  DR_BASE = {os.environ['DR_BASE']}")
print(f"  ✅ This session will use R: drive")

# Confirm R: is accessible
r = Path("R:/DR_Grading_v21")
print(f"  R: drive accessible: {Path('R:/').exists()}")

  DR_BASE = R:\DR_Grading_v21
  ✅ This session will use R: drive
  R: drive accessible: True


In [2]:
import shutil, os
from pathlib import Path

freed = 0

# ── 1. Delete DR project folder on C: ────────────────────────
old_c = Path.home() / "DR_Grading_v21"
if old_c.exists():
    size = sum(f.stat().st_size for f in old_c.rglob("*") if f.is_file())
    shutil.rmtree(str(old_c))
    freed += size
    print(f"  ✅ Deleted DR_Grading_v21 from C: ({size/1e9:.2f} GB freed)")
else:
    print("  ✅ No DR folder on C: drive")

# ── 2. Delete Kaggle download cache ───────────────────────────
kaggle_cache = Path.home() / ".kaggle" / "cache"
if kaggle_cache.exists():
    size = sum(f.stat().st_size for f in kaggle_cache.rglob("*") if f.is_file())
    shutil.rmtree(str(kaggle_cache))
    freed += size
    print(f"  ✅ Deleted Kaggle cache ({size/1e9:.2f} GB freed)")

# ── 3. Delete pip cache ───────────────────────────────────────
pip_cache = Path.home() / "AppData" / "Local" / "pip" / "cache"
if pip_cache.exists():
    size = sum(f.stat().st_size for f in pip_cache.rglob("*") if f.is_file())
    shutil.rmtree(str(pip_cache))
    freed += size
    print(f"  ✅ Deleted pip cache ({size/1e9:.2f} GB freed)")

# ── 4. Delete Jupyter notebook checkpoints ────────────────────
for cp in Path.home().rglob(".ipynb_checkpoints"):
    if cp.is_dir():
        size = sum(f.stat().st_size for f in cp.rglob("*") if f.is_file())
        shutil.rmtree(str(cp))
        freed += size
        print(f"  ✅ Deleted checkpoint: {cp} ({size/1e6:.1f} MB freed)")

# ── 5. Delete temp files ──────────────────────────────────────
temp = Path(os.environ.get("TEMP", "C:/Windows/Temp"))
if temp.exists():
    size = 0
    for f in temp.iterdir():
        try:
            if f.is_file():
                size += f.stat().st_size
                f.unlink()
            elif f.is_dir():
                size += sum(x.stat().st_size for x in f.rglob("*") if x.is_file())
                shutil.rmtree(str(f))
        except Exception:
            pass  # skip locked files
    freed += size
    print(f"  ✅ Deleted temp files ({size/1e9:.2f} GB freed)")

# ── 6. Delete Anaconda/pip downloaded packages cache ─────────
conda_pkgs = Path("C:/ProgramData/anaconda3/pkgs")
if conda_pkgs.exists():
    size = sum(f.stat().st_size for f in conda_pkgs.rglob("*") if f.is_file())
    os.system("conda clean --all -y")
    print(f"  ✅ Cleaned conda packages cache ({size/1e9:.2f} GB freed)")

print()
print(f"  💾 Total freed: {freed/1e9:.2f} GB")
print()
print("  ✅ Done! Now:")
print("  1. Close Jupyter fully (File → Shut Down)")
print("  2. Open new terminal as Administrator")
print("  3. Run: set DR_BASE=R:\\DR_Grading_v21")
print("  4. Run: jupyter notebook")

  ✅ No DR folder on C: drive
  ✅ Deleted checkpoint: C:\Users\RAKSHITA BAI.J\.ipynb_checkpoints (0.2 MB freed)
  ✅ Deleted temp files (0.03 GB freed)

  💾 Total freed: 0.03 GB

  ✅ Done! Now:
  1. Close Jupyter fully (File → Shut Down)
  2. Open new terminal as Administrator
  3. Run: set DR_BASE=R:\DR_Grading_v21
  4. Run: jupyter notebook


## Step 0 — Global Resume & Recovery System

In [3]:
# ═══════════════════════════════════════════════════════════════
# STEP 1 — SYSTEM SETUP + GPU DETECTION (ROBUST)
# Detects: NVIDIA CUDA → Apple MPS → CPU
# ═══════════════════════════════════════════════════════════════
import os, sys, json, pickle, time, platform, shutil, subprocess, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

# ── Bootstrap Step 0 if not already run ───────────────────────
HOME     = Path.home()
BASE_DIR = Path(os.environ.get("DR_BASE", str(HOME / "DR_Grading_v21")))
for name in ["data","flags","checkpoints","logs","cache",
             "artifacts","plots","export","deploy","state"]:
    (BASE_DIR / name).mkdir(parents=True, exist_ok=True)

DATA_DIR     = BASE_DIR / "data"
FLAG_DIR     = BASE_DIR / "flags"
CKPT_DIR     = BASE_DIR / "checkpoints"
LOG_DIR      = BASE_DIR / "logs"
CACHE_DIR    = BASE_DIR / "cache"
ARTIFACT_DIR = BASE_DIR / "artifacts"
PLOT_DIR     = BASE_DIR / "plots"
EXPORT_DIR   = BASE_DIR / "export"
DEPLOY_DIR   = BASE_DIR / "deploy"
STATE_DIR    = BASE_DIR / "state"
APTOS19_DIR  = DATA_DIR / "aptos2019"
DR2015_DIR   = DATA_DIR / "dr2015"
for d in [APTOS19_DIR, DR2015_DIR]: d.mkdir(parents=True, exist_ok=True)

NUM_CLASSES = 5; N_FOLDS = 5; SEED = 42
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
GRADE_MAP = {0:"No DR",1:"Mild",2:"Moderate",3:"Severe",4:"Proliferative"}

def is_done(s):   return (FLAG_DIR / f"{s}.done").exists()
def mark_done(s): (FLAG_DIR / f"{s}.done").touch()
def clear_done(s):
    f = FLAG_DIR / f"{s}.done"
    if f.exists(): f.unlink()
def save_json(data, path): Path(path).write_text(json.dumps(data, indent=2, default=str))
def load_json(path): return json.loads(Path(path).read_text())
def save_pickle(obj, path):
    with open(path,"wb") as f: pickle.dump(obj,f)
def load_pickle(path):
    with open(path,"rb") as f: return pickle.load(f)
STATE_FILE = STATE_DIR / "train_state.pkl"
def state_load():
    if STATE_FILE.exists():
        try: return load_pickle(STATE_FILE)
        except: return {}
    return {}
def state_save(key,val):
    s=state_load(); s[key]=val; save_pickle(s,STATE_FILE)
def state_get(key,default=None): return state_load().get(key,default)
def step_header(num,name):
    print("="*70); print(f"  STEP {num:02d} — {name}"); print("="*70)
    return time.time()
def step_skip(num,name,detail=""):
    print("="*70); print(f"  STEP {num:02d} — {name}")
    print(f"  ✅ ALREADY COMPLETED — Skipping")
    if detail: print(f"  Info: {detail}")
    print("="*70)
def step_done(num,t0,info=None):
    print(f"\n  ✅ Completed in {time.time()-t0:.1f}s")
    if info:
        for k,v in info.items(): print(f"  {k}: {v}")
    print("="*70)

# ── Now run Step 1 ────────────────────────────────────────────
if is_done("system"):
    info = load_json(LOG_DIR / "system_info.json")
    step_skip(1, "SYSTEM SETUP", f"Device: {info.get('device', '?')}")
else:
    t0 = step_header(1, "SYSTEM SETUP & GPU DETECTION")
    info = {
        "os": f"{platform.system()} {platform.release()}",
        "python": sys.version.split()[0],
        "arch": platform.machine(),
    }
    try:
        import psutil
        ram = psutil.virtual_memory()
        info["ram_total_gb"] = round(ram.total / 1e9, 1)
        info["ram_avail_gb"] = round(ram.available / 1e9, 1)
        print(f"  RAM: {info['ram_total_gb']} GB total, {info['ram_avail_gb']} GB available")
    except ImportError:
        print("  RAM: psutil not installed (optional)")

    _, _, free = shutil.disk_usage(str(HOME))
    info["disk_free_gb"] = round(free / 1e9, 1)
    print(f"  Disk: {info['disk_free_gb']} GB free")

    nvidia_gpu_found = False
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10
        )
        if r.returncode == 0 and r.stdout.strip():
            info["gpu_nvidia_smi"] = r.stdout.strip()
            nvidia_gpu_found = True
            print(f"  nvidia-smi: {info['gpu_nvidia_smi']}")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass

    import torch
    info["pytorch"] = torch.__version__

    if torch.cuda.is_available():
        info["device"] = "cuda"
        info["cuda_version"] = torch.version.cuda
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["vram_gb"] = round(torch.cuda.get_device_properties(0).total_memory / 1e9, 1)
        print(f"  PyTorch CUDA: ✅ {info['cuda_version']}")
        print(f"  GPU: {info['gpu_name']} ({info['vram_gb']} GB VRAM)")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        info["device"] = "mps"
        print(f"  Apple MPS: ✅ Available")
    else:
        info["device"] = "cpu"
        print(f"  Device: CPU only")
        if nvidia_gpu_found:
            print("  ⚠️ NVIDIA GPU found but CUDA not available!")
            print("  Fix: pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")

    import random, numpy as np
    def seed_everything(seed=SEED):
        random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
    seed_everything()
    info["seed"] = SEED

    save_json(info, LOG_DIR / "system_info.json")
    mark_done("system")
    step_done(1, t0, {"Device": info["device"], "PyTorch": info["pytorch"]})

  STEP 01 — SYSTEM SETUP
  ✅ ALREADY COMPLETED — Skipping
  Info: Device: cuda


## Step 1 — System Setup & GPU Detection

**Critical:** This cell properly detects CUDA GPUs (RTX 2050), Apple MPS, or CPU.
If CUDA is not detected but you have an NVIDIA GPU, the cell provides exact fix instructions.

In [4]:
# ═══════════════════════════════════════════════════════════════
# STEP 1 — SYSTEM SETUP + GPU DETECTION (ROBUST)
# Detects: NVIDIA CUDA → Apple MPS → CPU
# Provides fix instructions if GPU not detected
# ═══════════════════════════════════════════════════════════════
import sys, platform, shutil, subprocess

if is_done("system"):
    info = load_json(LOG_DIR / "system_info.json")
    step_skip(1, "SYSTEM SETUP", f"Device: {info.get('device', '?')}")
else:
    t0 = step_header(1, "SYSTEM SETUP & GPU DETECTION")

    info = {
        "os": f"{platform.system()} {platform.release()}",
        "python": sys.version.split()[0],
        "arch": platform.machine(),
    }

    # ── RAM ────────────────────────────────────────────────────
    try:
        import psutil
        ram = psutil.virtual_memory()
        info["ram_total_gb"] = round(ram.total / 1e9, 1)
        info["ram_avail_gb"] = round(ram.available / 1e9, 1)
        print(f"  RAM: {info['ram_total_gb']} GB total, {info['ram_avail_gb']} GB available")
    except ImportError:
        print("  RAM: psutil not installed (optional)")

    # ── Disk ───────────────────────────────────────────────────
    _, _, free = shutil.disk_usage(str(HOME))
    info["disk_free_gb"] = round(free / 1e9, 1)
    print(f"  Disk: {info['disk_free_gb']} GB free")

    # ── NVIDIA GPU check (system-level) ───────────────────────
    nvidia_gpu_found = False
    try:
        r = subprocess.run(
            ["nvidia-smi", "--query-gpu=name,memory.total,driver_version",
             "--format=csv,noheader"],
            capture_output=True, text=True, timeout=10
        )
        if r.returncode == 0 and r.stdout.strip():
            info["gpu_nvidia_smi"] = r.stdout.strip()
            nvidia_gpu_found = True
            print(f"  nvidia-smi: {info['gpu_nvidia_smi']}")
    except (FileNotFoundError, subprocess.TimeoutExpired):
        pass

    # ── PyTorch import + CUDA check ───────────────────────────
    import torch
    info["pytorch"] = torch.__version__

    if torch.cuda.is_available():
        info["device"] = "cuda"
        info["cuda_version"] = torch.version.cuda
        info["gpu_name"] = torch.cuda.get_device_name(0)
        info["vram_gb"] = round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1)
        print(f"  PyTorch CUDA: ✅ {info['cuda_version']}")
        print(f"  GPU: {info['gpu_name']} ({info['vram_gb']} GB VRAM)")
    elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        info["device"] = "mps"
        print(f"  Apple MPS: ✅ Available")
    else:
        info["device"] = "cpu"
        print(f"  Device: CPU only")

        # ── Diagnostic if NVIDIA GPU exists but CUDA not working ──
        if nvidia_gpu_found:
            print("\n" + "!" * 70)
            print("  ⚠️  NVIDIA GPU DETECTED but PyTorch CUDA is NOT available!")
            print("!" * 70)
            print(f"  PyTorch version: {torch.__version__}")
            print(f"  torch.cuda.is_available(): {torch.cuda.is_available()}")
            print(f"  PyTorch CUDA compiled: {torch.version.cuda}")
            print()
            print("  FIX — Install PyTorch with CUDA support:")
            print("  ─────────────────────────────────────────")
            print("  For CUDA 12.x (RTX 2050/3000/4000 series):")
            print("    pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")
            print()
            print("  For CUDA 11.8 (older systems):")
            print("    pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118")
            print()
            print("  Also verify:")
            print("  1. NVIDIA drivers are up to date (nvidia-smi shows driver)")
            print("  2. CUDA Toolkit is installed: https://developer.nvidia.com/cuda-downloads")
            print("  3. Environment variable CUDA_HOME or CUDA_PATH is set")
            print()
            print("  After installing, RESTART the kernel and re-run.")
            print("!" * 70)
        elif platform.system() == "Windows":
            print("\n  Tip: If you have an NVIDIA GPU (RTX 2050, etc.):")
            print("  pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121")

    # ── Reproducibility ───────────────────────────────────────
    import random, numpy as np
    def seed_everything(seed=SEED):
        random.seed(seed)
        np.random.seed(seed)
        torch.manual_seed(seed)
        os.environ["PYTHONHASHSEED"] = str(seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False

    seed_everything()
    info["seed"] = SEED

    save_json(info, LOG_DIR / "system_info.json")
    mark_done("system")
    step_done(1, t0, {"Device": info["device"], "PyTorch": info["pytorch"]})

  STEP 01 — SYSTEM SETUP
  ✅ ALREADY COMPLETED — Skipping
  Info: Device: cuda


## Step 2 — Install Requirements

In [5]:
# ═══════════════════════════════════════════════════════════════
# STEP 2 — INSTALL REQUIREMENTS (only installs what's missing)
# ═══════════════════════════════════════════════════════════════
import importlib

if is_done("install"):
    step_skip(2, "INSTALL REQUIREMENTS")
else:
    t0 = step_header(2, "INSTALL REQUIREMENTS")

    REQUIRED = {
        "torch": "torch>=2.1",
        "torchvision": "torchvision",
        "timm": "timm>=1.0.0",
        "albumentations": "albumentations>=1.4.0",
        "cv2": "opencv-python-headless",
        "sklearn": "scikit-learn",
        "scipy": "scipy",
        "pandas": "pandas",
        "numpy": "numpy",
        "tqdm": "tqdm",
        "matplotlib": "matplotlib",
        "PIL": "pillow<11.0",
        "psutil": "psutil",
        "kaggle": "kaggle",
        "huggingface_hub": "huggingface_hub>=0.23.0",
        "streamlit": "streamlit>=1.35.0",
    }

    missing = []
    for mod, pkg in REQUIRED.items():
        try:
            importlib.import_module(mod)
        except ImportError:
            missing.append(pkg)

    print(f"  Installed: {len(REQUIRED) - len(missing)}/{len(REQUIRED)}")
    print(f"  Missing:   {len(missing)}")

    if missing:
        for pkg in missing:
            print(f"  Installing {pkg}...")
            subprocess.run(
                [sys.executable, "-m", "pip", "install", "-q", "--upgrade", pkg],
                capture_output=True, text=True
            )
        print(f"  ✅ Installed {len(missing)} packages")
    else:
        print("  ✅ All packages present")

    mark_done("install")
    step_done(2, t0)

  STEP 02 — INSTALL REQUIREMENTS
  ✅ ALREADY COMPLETED — Skipping


## Step 3 — Kaggle Authentication

**Three methods supported (auto-tries in order):**
1. Existing `~/.kaggle/kaggle.json`
2. Environment variables `KAGGLE_USERNAME` + `KAGGLE_KEY`
3. Manual entry via `input()` prompt

**Get your API key:** https://www.kaggle.com/settings → API → Create New Token

In [6]:
# ═══════════════════════════════════════════════════════════════
# STEP 3 — KAGGLE AUTHENTICATION (robust, 3 methods)
# ═══════════════════════════════════════════════════════════════
if is_done("kaggle_auth"):
    kj = Path.home() / ".kaggle" / "kaggle.json"
    usr = "?"
    if kj.exists():
        try:
            usr = json.loads(kj.read_text()).get("username", "?")
        except Exception:
            pass
    step_skip(3, "KAGGLE AUTH", f"User: {usr}")
else:
    t0 = step_header(3, "KAGGLE AUTHENTICATION")

    KAGGLE_DIR = Path.home() / ".kaggle"
    KAGGLE_JSON = KAGGLE_DIR / "kaggle.json"

    configured = False

    # Method 1: Existing file
    if KAGGLE_JSON.exists():
        try:
            d = json.loads(KAGGLE_JSON.read_text())
            if d.get("username") and d.get("key"):
                print(f"  ✅ Found existing kaggle.json — User: {d['username']}")
                configured = True
        except Exception:
            pass

    # Method 2: Environment variables
    if not configured:
        ku = os.environ.get("KAGGLE_USERNAME", "")
        kk = os.environ.get("KAGGLE_KEY", "")
        if ku and kk:
            KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
            KAGGLE_JSON.write_text(json.dumps({"username": ku, "key": kk}))
            if platform.system() != "Windows":
                KAGGLE_JSON.chmod(0o600)
            print(f"  ✅ Written from environment variables — User: {ku}")
            configured = True

    # Method 3: Manual input
    if not configured:
        print("  No kaggle.json found and no environment variables set.")
        print("  Get your API key: https://www.kaggle.com/settings → API → Create New Token")
        print()
        try:
            ku = input("  Enter Kaggle username: ").strip()
            kk = input("  Enter Kaggle API key:  ").strip()
            if ku and kk:
                KAGGLE_DIR.mkdir(parents=True, exist_ok=True)
                KAGGLE_JSON.write_text(json.dumps({"username": ku, "key": kk}))
                if platform.system() != "Windows":
                    KAGGLE_JSON.chmod(0o600)
                print(f"  ✅ Saved — User: {ku}")
                configured = True
            else:
                print("  ⚠️ Empty input. Set KAGGLE_USERNAME and KAGGLE_KEY env vars,")
                print(f"     or place kaggle.json at: {KAGGLE_JSON}")
        except EOFError:
            print("  ⚠️ Non-interactive mode. Set env vars or place kaggle.json manually.")

    if configured:
        os.environ["KAGGLE_CONFIG_DIR"] = str(KAGGLE_DIR)

        # Verify authentication
        try:
            import kaggle
            kaggle.api.authenticate()
            print("  ✅ Kaggle API authenticated successfully")
        except Exception as e:
            print(f"  ⚠️ Auth test: {e}")
            print("  Download may still work — continuing...")

        mark_done("kaggle_auth")
        step_done(3, t0)
    else:
        print("  ⚠️ Configure Kaggle credentials, then re-run this cell.")
        print("=" * 70)

  STEP 03 — KAGGLE AUTH
  ✅ ALREADY COMPLETED — Skipping
  Info: User: karthickraja1111


## Step 4 — Download APTOS 2019 Dataset

Downloads from Kaggle competition. You must have accepted the competition rules:
https://www.kaggle.com/c/aptos2019-blindness-detection/rules

In [7]:
# ═══════════════════════════════════════════════════════════════
# STEP 4 — DOWNLOAD APTOS 2019 (resumable: separate download + extract)
# ═══════════════════════════════════════════════════════════════
import zipfile, shutil

# ── Delete corrupt zip files first ────────────────────────────
for z in APTOS19_DIR.glob("*.zip"):
    size_gb = z.stat().st_size / 1e9
    if size_gb < 5.0:  # corrupt if less than 5 GB
        print(f"  Deleting corrupt zip: {z.name} ({size_gb:.2f} GB)")
        z.unlink()

# ── Clear stale flags so it re-downloads ──────────────────────
clear_done("aptos19_download")
clear_done("aptos19_ready")

t0 = step_header(4, "DOWNLOAD APTOS 2019")

ZIP_NAME = "aptos2019-blindness-detection.zip"
ZIP_PATH = APTOS19_DIR / ZIP_NAME

# ── Download ──────────────────────────────────────────────────
if not ZIP_PATH.exists():
    print("  Downloading APTOS 2019 (~9.5 GB)...")
    print("  NOTE: You must accept competition rules first at:")
    print("  https://www.kaggle.com/c/aptos2019-blindness-detection/rules")
    print()

    success = False

    # Method 1: Python API
    try:
        import kaggle
        kaggle.api.authenticate()
        kaggle.api.competition_download_files(
            "aptos2019-blindness-detection",
            path=str(APTOS19_DIR),
            quiet=False
        )
        print("  ✅ Downloaded via Kaggle Python API")
        success = True
    except Exception as e:
        print(f"  ⚠️ Python API failed: {e}")

    # Method 2: CLI fallback
    if not success:
        import shutil as sh
        kaggle_exe = sh.which("kaggle") or sh.which("kaggle.exe")
        if kaggle_exe:
            print("  Trying CLI fallback...")
            r = subprocess.run(
                [kaggle_exe, "competitions", "download",
                 "-c", "aptos2019-blindness-detection",
                 "-p", str(APTOS19_DIR)],
                capture_output=True, text=True
            )
            if r.returncode == 0:
                success = True
                print("  ✅ Downloaded via Kaggle CLI")
            else:
                print(f"  ❌ CLI failed: {r.stderr[-500:]}")

    if not success:
        print("  ❌ All download methods failed.")
        print("  Manual download:")
        print("  1. Go to https://www.kaggle.com/c/aptos2019-blindness-detection/data")
        print(f"  2. Download and extract to: {APTOS19_DIR}")
        raise RuntimeError("Dataset download failed.")
else:
    size_gb = ZIP_PATH.stat().st_size / 1e9
    print(f"  ✅ ZIP already exists ({size_gb:.2f} GB)")

mark_done("aptos19_download")

# ── Extract ───────────────────────────────────────────────────
img_dir  = APTOS19_DIR / "train_images"
csv_path = APTOS19_DIR / "train.csv"

if not (img_dir.exists() and csv_path.exists()):
    if not ZIP_PATH.exists():
        zips = list(APTOS19_DIR.glob("*.zip"))
        if zips:
            ZIP_PATH = zips[0]
        else:
            raise FileNotFoundError(f"No zip found in {APTOS19_DIR}")

    # Verify zip is valid before extracting
    size_gb = ZIP_PATH.stat().st_size / 1e9
    if size_gb < 5.0:
        raise RuntimeError(f"ZIP too small ({size_gb:.2f} GB) — likely corrupt. Re-run this cell.")

    print(f"  Extracting {ZIP_PATH.name} ({size_gb:.2f} GB)...")
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        members = zf.namelist()
        for i, m in enumerate(members):
            zf.extract(m, APTOS19_DIR)
            if (i + 1) % max(1, len(members) // 20) == 0:
                pct = (i + 1) / len(members) * 100
                print(f"\r  Extracting: {pct:.0f}%", end="", flush=True)
        print()
else:
    print("  ✅ Already extracted")

n = len(list(img_dir.glob("*.png"))) if img_dir.exists() else 0
mark_done("aptos19_ready")
step_done(4, t0, {"Images": f"{n:,}", "Path": str(img_dir)})

  Deleting corrupt zip: aptos2019-blindness-detection.zip (4.97 GB)
  STEP 04 — DOWNLOAD APTOS 2019
  NOTE: You must accept competition rules first at:
  https://www.kaggle.com/c/aptos2019-blindness-detection/rules



100%|██████████| 9.51G/9.51G [10:04<00:00, 16.9MB/s]  



  ✅ Downloaded via Kaggle Python API
  Extracting aptos2019-blindness-detection.zip (10.22 GB)...
  Extracting: 100%

  ✅ Completed in 695.7s
  Images: 3,662
  Path: R:\DR_Grading_v21\data\aptos2019\train_images


## Step 5 — Download Diabetic Retinopathy 2015 Dataset

The 1st-place solution merged APTOS 2019 + DR 2015 (train + test) as training data.
Accept rules: https://www.kaggle.com/c/diabetic-retinopathy-detection/rules

In [8]:
# ═══════════════════════════════════════════════════════════════
# STEP 5 — DOWNLOAD DR 2015 (Stage 1: merged with APTOS 2019)
# ═══════════════════════════════════════════════════════════════
import zipfile

if is_done("dr2015_ready"):
    n_train = len(list((DR2015_DIR / "train").glob("*.jpeg"))) if (DR2015_DIR / "train").exists() else 0
    n_test = len(list((DR2015_DIR / "test").glob("*.jpeg"))) if (DR2015_DIR / "test").exists() else 0
    step_skip(5, "DOWNLOAD DR 2015", f"Train: {n_train:,}, Test: {n_test:,}")
else:
    t0 = step_header(5, "DOWNLOAD DR 2015")

    print("  NOTE: DR 2015 is ~85 GB. This is optional but recommended.")
    print("  Accept rules: https://www.kaggle.com/c/diabetic-retinopathy-detection/rules")
    print()

    try:
        import kaggle
        kaggle.api.authenticate()

        # Download train labels
        if not (DR2015_DIR / "trainLabels.csv").exists():
            print("  Downloading trainLabels.csv...")
            kaggle.api.competition_download_file(
                "diabetic-retinopathy-detection",
                file_name="trainLabels.csv.zip",
                path=str(DR2015_DIR),
                quiet=False
            )
            # Extract if zipped
            for z in DR2015_DIR.glob("trainLabels*.zip"):
                with zipfile.ZipFile(z, "r") as zf:
                    zf.extractall(DR2015_DIR)
                print("  ✅ trainLabels.csv extracted")

        # Download train images
        if not (DR2015_DIR / "train").exists() or len(list((DR2015_DIR / "train").glob("*.jpeg"))) < 100:
            print("  Downloading train images (~35 GB)...")
            print("  This will take a while...")
            try:
                kaggle.api.competition_download_file(
                    "diabetic-retinopathy-detection",
                    file_name="train.zip.001",
                    path=str(DR2015_DIR),
                    quiet=False
                )
            except Exception as e:
                print(f"  ⚠️ DR 2015 train download issue: {e}")
                print("  Continuing without DR 2015 — APTOS 2019 alone will still work.")
        else:
            print(f"  ✅ DR 2015 train images already present")

        # Download test labels + images (winner used test set too)
        if not (DR2015_DIR / "retinopathy_solution.csv").exists():
            print("  Downloading test labels...")
            try:
                kaggle.api.competition_download_file(
                    "diabetic-retinopathy-detection",
                    file_name="retinopathy_solution.csv.zip",
                    path=str(DR2015_DIR),
                    quiet=False
                )
                for z in DR2015_DIR.glob("retinopathy_solution*.zip"):
                    with zipfile.ZipFile(z, "r") as zf:
                        zf.extractall(DR2015_DIR)
            except Exception:
                pass

        mark_done("dr2015_ready")
        n_train = len(list((DR2015_DIR / "train").glob("*.jpeg"))) if (DR2015_DIR / "train").exists() else 0
        n_test = len(list((DR2015_DIR / "test").glob("*.jpeg"))) if (DR2015_DIR / "test").exists() else 0
        step_done(5, t0, {"Train": f"{n_train:,}", "Test": f"{n_test:,}"})

    except Exception as e:
        print(f"  ⚠️ DR 2015 download failed: {e}")
        print("  Continuing with APTOS 2019 only. Stage 1 will still work.")
        mark_done("dr2015_ready")
        step_done(5, t0, {"Status": "Skipped (APTOS 2019 only)"})

  STEP 05 — DOWNLOAD DR 2015
  NOTE: DR 2015 is ~85 GB. This is optional but recommended.
  Accept rules: https://www.kaggle.com/c/diabetic-retinopathy-detection/rules



100%|██████████| 69.4k/69.4k [00:00<00:00, 110kB/s]



  ✅ trainLabels.csv extracted
  This will take a while...


100%|██████████| 7.81G/7.81G [08:19<00:00, 16.8MB/s]  




  ✅ Completed in 506.8s
  Train: 0
  Test: 0


In [ ]:
# ── Force re-download DR 2015 ──────────────────────────────────
import subprocess, sys, zipfile

# Install kaggle if missing
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "kaggle"])
import kaggle

# Clear the stale flag
clear_done("dr2015_ready")

t0 = step_header(5, "DOWNLOAD DR 2015 — FORCED RETRY")

print("  ✅ Rules already accepted — proceeding with download...")
print()

kaggle.api.authenticate()

# Download labels
if not (DR2015_DIR / "trainLabels.csv").exists():
    print("  Downloading trainLabels.csv...")
    kaggle.api.competition_download_file(
        "diabetic-retinopathy-detection",
        file_name="trainLabels.csv.zip",
        path=str(DR2015_DIR),
        quiet=False
    )
    for z in DR2015_DIR.glob("trainLabels*.zip"):
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(DR2015_DIR)
        print("  ✅ trainLabels.csv extracted")
else:
    print("  ✅ trainLabels.csv already present")

# Download all DR 2015 files at once
print("  Downloading all DR 2015 files (~85 GB)...")
print("  This will take a LONG time — do NOT close Jupyter...")
try:
    kaggle.api.competition_download_files(
        "diabetic-retinopathy-detection",
        path=str(DR2015_DIR),
        quiet=False
    )
    print("  ✅ Download complete")
except Exception as e:
    print(f"  ❌ Failed: {e}")
    print()
    print("  MANUAL ALTERNATIVE:")
    print("  1. Go to https://www.kaggle.com/c/diabetic-retinopathy-detection/data")
    print("  2. Download train.zip.001–005 + trainLabels.csv.zip")
    print(f"  3. Place all files in: {DR2015_DIR}")
    print("  4. Re-run this cell")

# Extract all zips found
train_dir = DR2015_DIR / "train"
if not train_dir.exists() or len(list(train_dir.glob("*.jpeg"))) < 100:
    zips = sorted(DR2015_DIR.glob("*.zip"))
    for z in zips:
        print(f"  Extracting {z.name}...")
        try:
            with zipfile.ZipFile(z, "r") as zf:
                zf.extractall(DR2015_DIR)
        except Exception as e:
            print(f"  ⚠️ {z.name}: {e}")

n_train = len(list(train_dir.glob("*.jpeg"))) if train_dir.exists() else 0
n_test = len(list((DR2015_DIR / "test").glob("*.jpeg"))) if (DR2015_DIR / "test").exists() else 0
print(f"\n  Train images: {n_train:,}")
print(f"  Test images:  {n_test:,}")

if n_train > 0:
    mark_done("dr2015_ready")
    step_done(5, t0, {"Train": f"{n_train:,}", "Test": f"{n_test:,}"})
else:
    print("\n  ⚠️ Still 0 images — check competition rules acceptance and retry")

  STEP 05 — DOWNLOAD DR 2015 — FORCED RETRY
  ✅ Rules already accepted — proceeding with download...

  ✅ trainLabels.csv already present
  This will take a LONG time — do NOT close Jupyter...


  2%|▏         | 1.55G/82.2G [01:39<1:14:58, 19.3MB/s] 

In [ ]:
import shutil, os
from pathlib import Path

# ── Set new base on R: drive ───────────────────────────────────
NEW_BASE = Path("R:/DR_Grading_v21")
OLD_BASE = Path.home() / "DR_Grading_v21"

print(f"  Moving from: {OLD_BASE}")
print(f"  Moving to:   {NEW_BASE}")

# Create new base
NEW_BASE.mkdir(parents=True, exist_ok=True)

# Move everything
if OLD_BASE.exists():
    for item in OLD_BASE.iterdir():
        dest = NEW_BASE / item.name
        if dest.exists():
            print(f"  Skipping (already exists): {item.name}")
            continue
        print(f"  Moving: {item.name} ...")
        shutil.move(str(item), str(dest))
    print("  ✅ Move complete")
else:
    print("  ⚠️ Old base not found — may already be moved")

# Set environment variable permanently for future sessions
os.environ["DR_BASE"] = str(NEW_BASE)
print(f"\n  DR_BASE = {NEW_BASE}")
print("\n  ⚠️ Now set this permanently:")
print(f'  In Windows search: "Environment Variables"')
print(f'  → Add User Variable: DR_BASE = R:\\DR_Grading_v21')
print(f'  → Restart Jupyter after setting')

In [ ]:
import shutil, os
from pathlib import Path

# ── Set new base on R: drive ───────────────────────────────────
NEW_BASE = Path("R:/DR_Grading_v21")
OLD_BASE = Path.home() / "DR_Grading_v21"

print(f"  Moving from: {OLD_BASE}")
print(f"  Moving to:   {NEW_BASE}")

# Create new base
NEW_BASE.mkdir(parents=True, exist_ok=True)

# Move everything
if OLD_BASE.exists():
    for item in OLD_BASE.iterdir():
        dest = NEW_BASE / item.name
        if dest.exists():
            print(f"  Skipping (already exists): {item.name}")
            continue
        print(f"  Moving: {item.name} ...")
        shutil.move(str(item), str(dest))
    print("  ✅ Move complete")
else:
    print("  ⚠️ Old base not found — may already be moved")

# Set environment variable permanently for future sessions
os.environ["DR_BASE"] = str(NEW_BASE)
print(f"\n  DR_BASE = {NEW_BASE}")
print("\n  ⚠️ Now set this permanently:")
print(f'  In Windows search: "Environment Variables"')
print(f'  → Add User Variable: DR_BASE = R:\\DR_Grading_v21')
print(f'  → Restart Jupyter after setting')

In [ ]:
# ── Force re-download DR 2015 ──────────────────────────────────
import zipfile, subprocess, sys

# Clear the stale flag
clear_done("dr2015_ready")

t0 = step_header(5, "DOWNLOAD DR 2015 — FORCED RETRY")

print("  Step 1: Accept competition rules at:")
print("  https://www.kaggle.com/c/diabetic-retinopathy-detection/rules")
print("  (Must accept in browser while logged into Kaggle)")
print()

import kaggle
kaggle.api.authenticate()

# Download labels
if not (DR2015_DIR / "trainLabels.csv").exists():
    print("  Downloading trainLabels.csv...")
    kaggle.api.competition_download_file(
        "diabetic-retinopathy-detection",
        file_name="trainLabels.csv.zip",
        path=str(DR2015_DIR),
        quiet=False
    )
    for z in DR2015_DIR.glob("trainLabels*.zip"):
        with zipfile.ZipFile(z, "r") as zf:
            zf.extractall(DR2015_DIR)
        print("  ✅ trainLabels.csv extracted")
else:
    print("  ✅ trainLabels.csv already present")

# Download all files at once (more reliable than per-file)
print("  Downloading all DR 2015 files (~85 GB)...")
print("  This will take a LONG time on home internet...")
try:
    kaggle.api.competition_download_files(
        "diabetic-retinopathy-detection",
        path=str(DR2015_DIR),
        quiet=False
    )
    print("  ✅ Downloaded")
except Exception as e:
    print(f"  ❌ Failed: {e}")
    print()
    print("  MANUAL ALTERNATIVE:")
    print("  1. Go to https://www.kaggle.com/c/diabetic-retinopathy-detection/data")
    print("  2. Download train.zip.001 through train.zip.005 + trainLabels.csv.zip")
    print(f"  3. Place all files in: {DR2015_DIR}")
    print("  4. Re-run this cell")

# Extract zip parts if present
train_dir = DR2015_DIR / "train"
if not train_dir.exists() or len(list(train_dir.glob("*.jpeg"))) < 100:
    zips = sorted(DR2015_DIR.glob("*.zip"))
    for z in zips:
        print(f"  Extracting {z.name}...")
        try:
            with zipfile.ZipFile(z, "r") as zf:
                zf.extractall(DR2015_DIR)
        except Exception as e:
            print(f"  ⚠️ {z.name}: {e}")

n_train = len(list(train_dir.glob("*.jpeg"))) if train_dir.exists() else 0
n_test = len(list((DR2015_DIR / "test").glob("*.jpeg"))) if (DR2015_DIR / "test").exists() else 0
print(f"\n  Train images: {n_train:,}")
print(f"  Test images:  {n_test:,}")

if n_train > 0:
    mark_done("dr2015_ready")
    step_done(5, t0, {"Train": f"{n_train:,}", "Test": f"{n_test:,}"})
else:
    print("\n  ⚠️ Still 0 images — check competition rules acceptance and retry")

## Step 6 — Master Imports & Device Configuration

**Always run this cell after kernel restart.** Re-establishes all imports and device.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 6 — MASTER IMPORTS + DEVICE (always run after restart)
# ═══════════════════════════════════════════════════════════════
import subprocess, sys

def _pip(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

try:
    import cv2
except ImportError:
    _pip("opencv-python-headless")
    import cv2

try:
    import timm
except ImportError:
    _pip("timm")
    import timm

try:
    import albumentations
except ImportError:
    _pip("albumentations")
    import albumentations

import os, sys, json, pickle, time, platform, gc, warnings, random
from pathlib import Path
from copy import deepcopy
from collections import OrderedDict
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torch.nn.parameter import Parameter
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    cohen_kappa_score, accuracy_score, precision_score,
    recall_score, f1_score, classification_report, confusion_matrix
)
from scipy.optimize import minimize
warnings.filterwarnings("ignore")
t0 = step_header(6, "MASTER IMPORTS & DEVICE")
# ── Device Detection ──────────────────────────────────────────
if torch.cuda.is_available():
    DEVICE = "cuda"
    USE_AMP = True
    print(f"  Device: CUDA — {torch.cuda.get_device_name(0)}")
    print(f"  VRAM:   {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = "mps"
    USE_AMP = False
    print(f"  Device: Apple MPS")
else:
    DEVICE = "cpu"
    USE_AMP = False
    print(f"  Device: CPU")
print(f"  AMP:    {'ON' if USE_AMP else 'OFF'}")
print(f"  PyTorch: {torch.__version__} | timm: {timm.__version__}")
# ── Reproducibility ───────────────────────────────────────────
def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
seed_everything()
# ── Safe torch.load ───────────────────────────────────────────
def safe_load(path, map_location="cpu"):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)
# ── QWK helper ────────────────────────────────────────────────
def qwk(y_true, y_pred):
    return cohen_kappa_score(y_true, y_pred, weights="quadratic")
step_done(6, t0)

## Step 7 — Load & Merge Datasets (APTOS 2019 + DR 2015)

Winner strategy: combine all 2015 (train+test) and 2019 data as training set.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 7 — LOAD & MERGE DATASETS
# Winner: merge APTOS 2019 + DR 2015 (train + test) for training
# ═══════════════════════════════════════════════════════════════
if is_done("data_merged"):
    df = pd.read_parquet(ARTIFACT_DIR / "df_merged.parquet")
    step_skip(7, "LOAD & MERGE DATASETS", f"{len(df):,} total images")
else:
    t0 = step_header(7, "LOAD & MERGE DATASETS")

    frames = []

    # ── APTOS 2019 ────────────────────────────────────────────
    csv_2019 = APTOS19_DIR / "train.csv"
    img_dir_2019 = APTOS19_DIR / "train_images"

    if csv_2019.exists():
        df_2019 = pd.read_csv(csv_2019)
        df_2019["image_path"] = df_2019["id_code"].apply(
            lambda x: str(img_dir_2019 / f"{x}.png")
        )
        df_2019["source"] = "aptos2019"
        df_2019 = df_2019.rename(columns={"diagnosis": "label"})
        # Keep only existing images
        df_2019 = df_2019[df_2019["image_path"].apply(
            lambda p: Path(p).exists()
        )].reset_index(drop=True)
        frames.append(df_2019[["id_code", "label", "image_path", "source"]])
        print(f"  APTOS 2019: {len(df_2019):,} images")
    else:
        raise FileNotFoundError(f"APTOS 2019 CSV not found at {csv_2019}. Run Step 4.")

    # ── DR 2015 Train ─────────────────────────────────────────
    csv_2015_train = DR2015_DIR / "trainLabels.csv"
    img_dir_2015_train = DR2015_DIR / "train"

    if csv_2015_train.exists() and img_dir_2015_train.exists():
        df_2015 = pd.read_csv(csv_2015_train)
        df_2015["image_path"] = df_2015["image"].apply(
            lambda x: str(img_dir_2015_train / f"{x}.jpeg")
        )
        df_2015["source"] = "dr2015_train"
        df_2015 = df_2015.rename(columns={"image": "id_code", "level": "label"})
        df_2015 = df_2015[df_2015["image_path"].apply(
            lambda p: Path(p).exists()
        )].reset_index(drop=True)
        frames.append(df_2015[["id_code", "label", "image_path", "source"]])
        print(f"  DR 2015 Train: {len(df_2015):,} images")
    else:
        print("  DR 2015 Train: not available (continuing without)")

    # ── DR 2015 Test ──────────────────────────────────────────
    csv_2015_test = DR2015_DIR / "retinopathy_solution.csv"
    img_dir_2015_test = DR2015_DIR / "test"

    if csv_2015_test.exists() and img_dir_2015_test.exists():
        df_2015t = pd.read_csv(csv_2015_test)
        # Solution CSV may have 'Usage' column — keep only Public+Private
        if "Usage" in df_2015t.columns:
            df_2015t = df_2015t[df_2015t["Usage"].isin(["Public", "Private"])]
        df_2015t["image_path"] = df_2015t["image"].apply(
            lambda x: str(img_dir_2015_test / f"{x}.jpeg")
        )
        df_2015t["source"] = "dr2015_test"
        df_2015t = df_2015t.rename(columns={"image": "id_code", "level": "label"})
        df_2015t = df_2015t[df_2015t["image_path"].apply(
            lambda p: Path(p).exists()
        )].reset_index(drop=True)
        frames.append(df_2015t[["id_code", "label", "image_path", "source"]])
        print(f"  DR 2015 Test:  {len(df_2015t):,} images")
    else:
        print("  DR 2015 Test:  not available (continuing without)")

    # ── Merge ─────────────────────────────────────────────────
    df = pd.concat(frames, ignore_index=True)
    df["label"] = df["label"].astype(int)
    df["grade_name"] = df["label"].map(GRADE_MAP)

    # Distribution
    print(f"\n  Total merged dataset: {len(df):,} images")
    for g in range(5):
        n = (df["label"] == g).sum()
        bar = "█" * (n // 200 + 1)
        print(f"    G{g} ({GRADE_MAP[g]:14s}): {n:6,} {bar}")

    df.to_parquet(ARTIFACT_DIR / "df_merged.parquet", index=False)
    mark_done("data_merged")
    step_done(7, t0, {"Total": f"{len(df):,}", "Sources": df["source"].nunique()})

## Step 8 — Data Cleaning (remove corrupt/black images only)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 8 — DATA CLEANING (conservative — only remove truly broken)
# ═══════════════════════════════════════════════════════════════
if is_done("cleaning"):
    df = pd.read_parquet(ARTIFACT_DIR / "df_clean.parquet")
    step_skip(8, "DATA CLEANING", f"{len(df):,} clean images")
else:
    t0 = step_header(8, "DATA CLEANING")

    def check_image(path):
        """Only reject truly broken images."""
        try:
            bgr = cv2.imread(str(path))
            if bgr is None:
                return False, "unreadable"
            h, w = bgr.shape[:2]
            if h < 50 or w < 50:
                return False, "too_small"
            gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
            if float(gray.mean()) < 3.0:
                return False, "black"
            return True, "ok"
        except Exception:
            return False, "error"

    results = []
    total = len(df)
    for i, (_, row) in enumerate(df.iterrows()):
        valid, reason = check_image(row["image_path"])
        results.append(valid)
        if (i + 1) % max(1, total // 20) == 0:
            print(f"\r  Checking: {(i+1)/total*100:.0f}%", end="", flush=True)
    print()

    n_removed = total - sum(results)
    df = df[results].reset_index(drop=True)
    print(f"  Removed: {n_removed} | Remaining: {len(df):,}")

    df.to_parquet(ARTIFACT_DIR / "df_clean.parquet", index=False)
    mark_done("cleaning")
    step_done(8, t0, {"Clean": f"{len(df):,}", "Removed": n_removed})

## Step 9 — Stratified K-Fold Split (5 folds)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 9 — STRATIFIED K-FOLD (5 folds on APTOS 2019 only)
# DR 2015 always goes to training; APTOS 2019 gets fold assignment
# ═══════════════════════════════════════════════════════════════
if is_done("kfold"):
    df = pd.read_parquet(ARTIFACT_DIR / "df_kfold.parquet")
    step_skip(9, "K-FOLD SPLIT", f"{N_FOLDS} folds, {len(df):,} rows")
else:
    t0 = step_header(9, "STRATIFIED K-FOLD SPLIT")

    # Fold assignment: only on APTOS 2019
    # DR 2015 data always trains (fold = -1, never in validation)
    df["fold"] = -1  # Default: always train

    mask_2019 = df["source"] == "aptos2019"
    idx_2019 = df[mask_2019].index
    labels_2019 = df.loc[mask_2019, "label"].values

    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    for fi, (_, vi) in enumerate(skf.split(idx_2019, labels_2019)):
        df.loc[idx_2019[vi], "fold"] = fi

    # Report
    for fold in range(N_FOLDS):
        n_val = (df["fold"] == fold).sum()
        n_train = (df["fold"] != fold).sum()
        print(f"  Fold {fold}: train={n_train:,} | val={n_val:,}")

    df.to_parquet(ARTIFACT_DIR / "df_kfold.parquet", index=False)
    mark_done("kfold")
    step_done(9, t0)

## Step 10 — Preprocessing Pipeline (MINIMAL — Winner Rule)

**From the 1st place solution:**
> *"I don't think it's necessary to preprocess images to help with the modelling,
> the image qualities are perfect as input for deep neural networks.
> So, no special preprocessing, just plain resizing."*

**Only:** Resize to target size. **No** CLAHE, no Ben Graham, no retina crop.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 10 — PREPROCESSING (MINIMAL — just resize)
# Winner: "no special preprocessing, just plain resizing"
# ═══════════════════════════════════════════════════════════════
t0 = step_header(10, "PREPROCESSING PIPELINE (MINIMAL)")

IMG_SIZE = 512  # Winner used 512 for best models

def preprocess_image(path_or_arr, size=IMG_SIZE):
    """Minimal preprocessing: read + resize. No CLAHE, no enhancement."""
    if isinstance(path_or_arr, np.ndarray):
        rgb = path_or_arr.copy()
    else:
        bgr = cv2.imread(str(path_or_arr))
        if bgr is None:
            return np.zeros((size, size, 3), np.uint8)
        rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    # Just resize — that's it
    rgb = cv2.resize(rgb, (size, size), interpolation=cv2.INTER_AREA)
    return rgb

# Quick benchmark
if len(df) > 0:
    _t = time.time()
    for _ in range(5):
        preprocess_image(df["image_path"].iloc[0])
    lat = (time.time() - _t) / 5 * 1000
    print(f"  Latency: {lat:.1f} ms/image")
    print(f"  IMG_SIZE: {IMG_SIZE}")
    print(f"  Method: resize only (winner strategy)")

step_done(10, t0)

## Step 11 — Augmentation, Dataset & DataLoader

Winner augmentations: HorizontalFlip, Rotate(±180°), ShiftScaleRotate,
BrightnessContrast, HueSaturation, Blur/Sharpen

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 11 — AUGMENTATION + DATASET + DATALOADER
# Winner augmentations faithfully replicated
# ═══════════════════════════════════════════════════════════════
t0 = step_header(11, "AUGMENTATION + DATASET PIPELINE")

def build_train_transforms(sz=IMG_SIZE):
    """Winner augmentations: contrast, brightness, hue, sat, blur, sharpen, rotate, mirror."""
    return A.Compose([
        A.Resize(sz, sz),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.Rotate(limit=180, p=0.7),
        A.ShiftScaleRotate(shift_limit=0.2, scale_limit=0.2, rotate_limit=0, p=0.5),
        A.RandomBrightnessContrast(brightness_limit=20/255.0, contrast_limit=0.2, p=0.5),
        A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20, p=0.5),
        A.OneOf([
            A.GaussianBlur(blur_limit=(3, 5), p=1.0),
            A.Sharpen(alpha=(0.2, 0.5), lightness=(0.5, 1.0), p=1.0),
        ], p=0.3),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def build_val_transforms(sz=IMG_SIZE):
    return A.Compose([
        A.Resize(sz, sz),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

def build_tta_transforms(sz=IMG_SIZE):
    """5 TTA views: original, hflip, +10°, -10°, vflip."""
    norm = A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
    resize = A.Resize(sz, sz)
    return [
        build_val_transforms(sz),
        A.Compose([resize, A.HorizontalFlip(p=1), norm, ToTensorV2()]),
        A.Compose([resize, A.Rotate(limit=(10, 10), p=1), norm, ToTensorV2()]),
        A.Compose([resize, A.Rotate(limit=(-10, -10), p=1), norm, ToTensorV2()]),
        A.Compose([resize, A.VerticalFlip(p=1), norm, ToTensorV2()]),
    ]


class DRDataset(Dataset):
    """Dataset for DR grading. Returns image tensor + regression label (float)."""

    def __init__(self, dataframe, transform=None, img_size=IMG_SIZE):
        self.df = dataframe.reset_index(drop=True)
        self.transform = transform
        self.img_size = img_size

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        label = float(row["label"])  # Regression target for SmoothL1Loss

        img = preprocess_image(row["image_path"], size=self.img_size)

        if self.transform:
            img = self.transform(image=img)["image"]
        else:
            img = torch.from_numpy(img.transpose(2, 0, 1)).float() / 255.0

        return img, torch.tensor(label, dtype=torch.float32)


def build_sampler(dataframe):
    """Weighted sampler for class imbalance."""
    labels = dataframe["label"].values
    counts = np.bincount(labels, minlength=5).astype(float)
    weights = 1.0 / np.maximum(counts, 1)
    sample_weights = weights[labels]
    return WeightedRandomSampler(
        torch.DoubleTensor(sample_weights), len(sample_weights), replacement=True
    )


def make_loader(dataset, batch_size=16, shuffle=True, sampler=None, drop_last=False):
    if sampler:
        shuffle = False
    nw = 0 if platform.system() == "Windows" else min(4, os.cpu_count() or 1)
    return DataLoader(
        dataset, batch_size=batch_size, shuffle=shuffle, sampler=sampler,
        num_workers=nw, pin_memory=(DEVICE == "cuda"),
        persistent_workers=(nw > 0), drop_last=drop_last
    )


# ── Batch size map (RTX 2050 ~4GB VRAM safe) ─────────────────
BS_MAP = {224: 16, 384: 8, 512: 2}
GRAD_ACCUM_MAP = {224: 1, 384: 2, 512: 8}  # Effective batch = BS × ACCUM

print(f"  Train augmentations: {len(build_train_transforms(224).transforms)} ops")
print(f"  TTA views: {len(build_tta_transforms(224))}")
print(f"  Batch sizes: {BS_MAP}")
print(f"  Grad accumulation: {GRAD_ACCUM_MAP}")

step_done(11, t0)

## Step 12 — Model Architecture (GeM + SmoothL1Loss Regression)

**Winner architecture:**
- Backbone (pretrained, `num_classes=0`) → GeM pooling → Linear(256) → BN → ReLU → Dropout(0.5) → Linear(1)
- **Output: single regression value** (not classification)
- **Loss: SmoothL1Loss** (not CrossEntropy)
- **GeM pooling** with learnable parameter p

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 12 — MODEL ARCHITECTURE
# GeM pooling + SmoothL1Loss regression (single output)
# ═══════════════════════════════════════════════════════════════
t0 = step_header(12, "MODEL ARCHITECTURE")

# ── GeM Pooling (from winner, learnable p) ────────────────────
class GeM(nn.Module):
    """Generalized Mean Pooling — https://arxiv.org/pdf/1711.02512.pdf"""
    def __init__(self, p=3, eps=1e-6):
        super().__init__()
        self.p = Parameter(torch.ones(1) * p)
        self.eps = eps

    def forward(self, x):
        return F.avg_pool2d(
            x.clamp(min=self.eps).pow(self.p),
            (x.size(-2), x.size(-1))
        ).pow(1.0 / self.p)

    def __repr__(self):
        return (f"{self.__class__.__name__}("
                f"p={self.p.data.tolist()[0]:.4f}, eps={self.eps})")


# ── DR Model ─────────────────────────────────────────────────
class DRModel(nn.Module):
    """
    Winner architecture:
    Backbone → GeM → Linear(256) → BN → ReLU → Dropout → Linear(1)
    Single regression output for SmoothL1Loss.
    """
    def __init__(self, backbone_name, pretrained=True, drop=0.5):
        super().__init__()
        self.backbone = timm.create_model(
            backbone_name, pretrained=pretrained,
            num_classes=0, global_pool=""
        )
        feat_dim = self.backbone.num_features
        self.pool = GeM()
        self.head = nn.Sequential(
            nn.Flatten(),
            nn.Linear(feat_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=True),
            nn.Dropout(drop),
            nn.Linear(256, 1),  # Single regression output
        )

    def forward(self, x):
        features = self.backbone(x)
        pooled = self.pool(features)
        return self.head(pooled).squeeze(-1)  # Shape: (batch,)

    def freeze_backbone(self):
        for p in self.backbone.parameters():
            p.requires_grad_(False)

    def unfreeze_all(self):
        for p in self.parameters():
            p.requires_grad_(True)


# ── EMA (Exponential Moving Average) ─────────────────────────
class EMA:
    """EMA with decay=0.9999 as specified."""
    def __init__(self, model, decay=0.9999):
        self.decay = decay
        self.shadow = {}
        self.backup = {}
        for name, param in model.named_parameters():
            if param.requires_grad:
                self.shadow[name] = param.data.clone()

    def update(self, model):
        for name, param in model.named_parameters():
            if param.requires_grad and name in self.shadow:
                self.shadow[name].mul_(self.decay).add_(param.data, alpha=1 - self.decay)

    def apply(self, model):
        """Apply EMA weights (save originals for restore)."""
        self.backup = {}
        for name, param in model.named_parameters():
            if name in self.shadow:
                self.backup[name] = param.data.clone()
                param.data.copy_(self.shadow[name])

    def restore(self, model):
        """Restore original weights."""
        for name, param in model.named_parameters():
            if name in self.backup:
                param.data.copy_(self.backup[name])
        self.backup = {}

    def state_dict(self):
        return {"shadow": self.shadow, "decay": self.decay}

    def load_state_dict(self, state):
        self.shadow = state["shadow"]
        self.decay = state.get("decay", self.decay)


# ── Model Configurations (8 models = 4 backbones × 2 seeds) ──
MODEL_CONFIGS = [
    {"name": "efficientnet_b4",      "backbone": "tf_efficientnet_b4",     "seeds": [42, 137]},
    {"name": "efficientnetv2_b1",    "backbone": "tf_efficientnetv2_b1",   "seeds": [42, 137]},
    {"name": "efficientnet_b3",      "backbone": "tf_efficientnet_b3",     "seeds": [42, 137]},
    {"name": "seresnext50",          "backbone": "seresnext50_32x4d",      "seeds": [42, 137]},
]

def build_model(backbone_name, pretrained=True):
    return DRModel(backbone_name, pretrained=pretrained).to(DEVICE)

# ── Verify one model builds correctly ─────────────────────────
test_model = build_model("tf_efficientnet_b3", pretrained=False)
n_params = sum(p.numel() for p in test_model.parameters()) / 1e6
print(f"  Test build (efficientnet_b3): {n_params:.2f}M params")
print(f"  Pool: {test_model.pool}")
print(f"  Output dim: 1 (regression)")
print(f"  Loss: SmoothL1Loss")
print(f"  EMA decay: 0.9999")
print(f"  Models in ensemble: {sum(len(c['seeds']) for c in MODEL_CONFIGS)}")
del test_model
gc.collect()

step_done(12, t0)

## Step 13 — DataLoader Sanity Check

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 13 — DATALOADER SANITY CHECK
# ═══════════════════════════════════════════════════════════════
if is_done("sanity_check"):
    step_skip(13, "DATALOADER SANITY CHECK")
else:
    t0 = step_header(13, "DATALOADER SANITY CHECK")

    _ds = DRDataset(df.head(64), transform=build_val_transforms(224), img_size=224)
    _dl = DataLoader(_ds, batch_size=8, shuffle=True, num_workers=0)

    imgs, labels = next(iter(_dl))
    print(f"  Batch shape: {imgs.shape}")
    print(f"  Labels (float): {labels[:5].tolist()}")
    print(f"  Dtype: {imgs.dtype}")
    assert imgs.shape == (8, 3, 224, 224), f"Shape mismatch: {imgs.shape}"
    assert imgs.dtype == torch.float32
    assert labels.dtype == torch.float32, "Labels should be float for regression"

    del _ds, _dl, imgs, labels
    gc.collect()

    mark_done("sanity_check")
    step_done(13, t0)

## Step 14 — Stage 1 Training (5-Fold × 3-Phase × 8 Models)

**Fully resumable:** saves state per fold/phase/epoch. Crash → re-run → continues.

**3 phases per model (progressive resizing):**
- Phase 1: 224px, frozen backbone, 10 epochs
- Phase 2: 384px, full finetune, 15 epochs
- Phase 3: 512px, full finetune, 10 epochs

**8 models** (4 backbones × 2 seeds) trained sequentially.

**Loss: SmoothL1Loss | Optimizer: AdamW | Scheduler: Warmup + Cosine | EMA: 0.9999**

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 14 — STAGE 1 TRAINING
# 5-Fold × 3-Phase × 8 Models (4 backbones × 2 seeds)
# FULLY RESUMABLE: per-model, per-fold, per-phase checkpoint
# ═══════════════════════════════════════════════════════════════

# ── Training Hyperparameters ──────────────────────────────────
PHASES = [
    {"size": 224, "epochs": 10, "name": "P1-Freeze",  "freeze": True,  "lr": 3e-4},
    {"size": 384, "epochs": 15, "name": "P2-Finetune", "freeze": False, "lr": 1e-4},
    {"size": 512, "epochs": 10, "name": "P3-FullRes",  "freeze": False, "lr": 3e-5},
]
WD = 1e-4
ES_PATIENCE = 5
ES_DELTA = 0.002

if is_done("stage1_training"):
    step_skip(14, "STAGE 1 TRAINING", "All models trained")
else:
    t0 = step_header(14, "STAGE 1 TRAINING")

    # ── Train one model+seed+fold ─────────────────────────────
    def train_one_fold(backbone_name, model_name, seed_val, fold):
        """Train a single fold of a single model. Returns best_qwk and OOF preds."""
        tag = f"{model_name}_s{seed_val}_f{fold}"
        fold_ckpt = CKPT_DIR / f"{tag}_best.pt"
        fold_oof = ARTIFACT_DIR / f"{tag}_oof.npy"

        # Already done?
        if is_done(tag) and fold_ckpt.exists():
            prev = safe_load(fold_ckpt, "cpu")
            print(f"    ✅ {tag}: QWK={prev.get('val_qwk', 0):.4f} (cached)")
            if fold_oof.exists():
                return prev.get("val_qwk", 0), np.load(str(fold_oof))
            return prev.get("val_qwk", 0), None

        seed_everything(seed_val + fold)

        # Splits
        df_train = df[df["fold"] != fold].reset_index(drop=True)
        df_val = df[df["fold"] == fold].reset_index(drop=True)

        # Build model
        model = build_model(backbone_name, pretrained=True)
        ema = EMA(model, decay=0.9999)
        criterion = nn.SmoothL1Loss()
        best_qwk = -1.0
        best_state = None

        for pi, phase in enumerate(PHASES):
            sz = phase["size"]
            nep = phase["epochs"]
            pn = phase["name"]
            bs = BS_MAP[sz]
            accum = GRAD_ACCUM_MAP[sz]
            plr = phase["lr"]

            p_ckpt = CKPT_DIR / f"{tag}_p{pi}.pt"

            # Freeze/unfreeze
            if phase["freeze"]:
                model.freeze_backbone()
            else:
                model.unfreeze_all()

            trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
            print(f"    [{pn}] {sz}px ×{nep}ep | {trainable/1e6:.1f}M params | bs={bs}×{accum}")

            # Datasets
            tr_ds = DRDataset(df_train, transform=build_train_transforms(sz), img_size=sz)
            va_ds = DRDataset(df_val, transform=build_val_transforms(sz), img_size=sz)
            smp = build_sampler(df_train)
            tr_dl = make_loader(tr_ds, bs, sampler=smp, drop_last=True)
            va_dl = make_loader(va_ds, bs, shuffle=False)

            # Optimizer + Scheduler
            opt = torch.optim.AdamW([
                {"params": model.head.parameters(), "lr": plr},
                {"params": model.pool.parameters(), "lr": plr},
                {"params": [p for p in model.backbone.parameters() if p.requires_grad],
                 "lr": plr / 10},
            ], weight_decay=WD)

            warmup_ep = min(2, nep // 5)
            def lr_lambda(ep, _we=warmup_ep, _ne=nep):
                if ep < _we:
                    return (ep + 1) / _we
                return 0.5 * (1 + np.cos(np.pi * (ep - _we) / max(_ne - _we, 1)))
            sched = torch.optim.lr_scheduler.LambdaLR(opt, lr_lambda)

            # Resume from phase checkpoint
            ep_start = 0
            if p_ckpt.exists():
                ps = safe_load(p_ckpt, DEVICE)
                model.load_state_dict(ps["model_state"])
                opt.load_state_dict(ps["optimizer_state"])
                sched.load_state_dict(ps["scheduler_state"])
                ep_start = ps["epoch"]
                best_qwk = ps.get("best_qwk", best_qwk)
                if ps.get("best_state"):
                    best_state = ps["best_state"]
                if ps.get("ema"):
                    ema.load_state_dict(ps["ema"])
                print(f"      ↻ Resuming from epoch {ep_start + 1}/{nep}")

            model.to(DEVICE)
            scaler = torch.amp.GradScaler("cuda") if USE_AMP else None
            es_cnt = 0

            for ep in range(ep_start, nep):
                # ── Train ─────────────────────────────────────
                model.train()
                epoch_loss = 0.0
                opt.zero_grad()

                for step, (imgs, labels) in enumerate(tr_dl):
                    imgs = imgs.to(DEVICE)
                    labels = labels.to(DEVICE)

                    if scaler:
                        with torch.amp.autocast("cuda"):
                            preds = model(imgs)
                            loss = criterion(preds, labels) / accum
                        scaler.scale(loss).backward()
                        if (step + 1) % accum == 0:
                            scaler.unscale_(opt)
                            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                            scaler.step(opt)
                            scaler.update()
                            opt.zero_grad()
                    else:
                        preds = model(imgs)
                        loss = criterion(preds, labels) / accum
                        loss.backward()
                        if (step + 1) % accum == 0:
                            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                            opt.step()
                            opt.zero_grad()

                    epoch_loss += loss.item() * accum
                    ema.update(model)

                sched.step()
                train_loss = epoch_loss / max(len(tr_dl), 1)

                # ── Validate (with EMA weights) ───────────────
                ema.apply(model)
                model.eval()
                val_preds_raw = []
                val_labels_list = []

                with torch.no_grad():
                    for imgs, labels in va_dl:
                        imgs = imgs.to(DEVICE)
                        out = model(imgs)
                        val_preds_raw.extend(out.cpu().numpy().tolist())
                        val_labels_list.extend(labels.numpy().astype(int).tolist())

                # Convert regression output to classes using thresholds
                val_preds_clipped = np.clip(val_preds_raw, 0, 4)
                val_preds_cls = np.round(val_preds_clipped).astype(int)
                val_preds_cls = np.clip(val_preds_cls, 0, 4)
                val_qwk = qwk(val_labels_list, val_preds_cls)

                ema.restore(model)

                # ── Best model tracking ───────────────────────
                flag = ""
                if val_qwk > best_qwk + ES_DELTA:
                    best_qwk = val_qwk
                    best_state = deepcopy(model.state_dict())
                    es_cnt = 0
                    flag = " ★ BEST"
                else:
                    es_cnt += 1

                lr_now = opt.param_groups[0]["lr"]
                print(f"      Ep{ep+1:02d} Loss={train_loss:.4f} QWK={val_qwk:.4f}{flag} ES={es_cnt}/{ES_PATIENCE} lr={lr_now:.1e}")

                # Save phase checkpoint
                ckpt_data = {
                    "epoch": ep + 1,
                    "model_state": model.state_dict(),
                    "best_state": best_state,
                    "optimizer_state": opt.state_dict(),
                    "scheduler_state": sched.state_dict(),
                    "best_qwk": best_qwk,
                    "ema": ema.state_dict(),
                }
                if scaler:
                    ckpt_data["scaler_state"] = scaler.state_dict()
                torch.save(ckpt_data, p_ckpt)

                if es_cnt >= ES_PATIENCE:
                    print(f"      ⏹ Early stop at epoch {ep + 1}")
                    break

            # Cleanup phase
            del tr_ds, va_ds, tr_dl, va_dl, opt, sched, scaler
            gc.collect()
            if DEVICE == "cuda":
                torch.cuda.empty_cache()

        # ── OOF predictions with TTA + EMA ────────────────────
        if best_state:
            model.load_state_dict(best_state)
        ema.apply(model)
        model.eval()

        oof_raw = np.zeros(len(df_val), dtype=np.float32)
        tta_tfs = build_tta_transforms(384)

        with torch.no_grad():
            for ttf in tta_tfs:
                va_tta = DRDataset(df_val, transform=ttf, img_size=384)
                ld_tta = make_loader(va_tta, BS_MAP[384], shuffle=False)
                batch_preds = []
                for imgs, _ in ld_tta:
                    out = model(imgs.to(DEVICE))
                    batch_preds.extend(out.cpu().numpy().tolist())
                oof_raw += np.array(batch_preds)
            oof_raw /= len(tta_tfs)

        ema.restore(model)

        # Save OOF and best model
        np.save(str(fold_oof), oof_raw)
        torch.save({
            "model_state": best_state if best_state else model.state_dict(),
            "val_qwk": best_qwk,
            "backbone": backbone_name,
            "ema": ema.state_dict(),
        }, fold_ckpt)

        mark_done(tag)
        print(f"    ✅ {tag} done — QWK={best_qwk:.4f}")

        del model
        gc.collect()
        if DEVICE == "cuda":
            torch.cuda.empty_cache()

        return best_qwk, oof_raw

    # ── Train ALL models ──────────────────────────────────────
    all_results = {}

    for config in MODEL_CONFIGS:
        backbone = config["backbone"]
        name = config["name"]

        for seed_val in config["seeds"]:
            model_tag = f"{name}_s{seed_val}"
            print(f"\n  {'═' * 25} {model_tag} {'═' * 25}")

            fold_qwks = []
            for fold in range(N_FOLDS):
                print(f"\n  ── Fold {fold} ──")
                best_q, oof = train_one_fold(backbone, name, seed_val, fold)
                fold_qwks.append(best_q)

            mean_qwk = np.mean(fold_qwks)
            print(f"\n  {model_tag} Mean QWK: {mean_qwk:.4f} ± {np.std(fold_qwks):.4f}")
            all_results[model_tag] = {
                "fold_qwks": fold_qwks,
                "mean_qwk": float(mean_qwk),
            }

    save_json(all_results, ARTIFACT_DIR / "stage1_results.json")
    mark_done("stage1_training")

    # Summary
    print("\n" + "=" * 70)
    print("  STAGE 1 SUMMARY")
    print("=" * 70)
    for tag, res in all_results.items():
        print(f"  {tag:30s}: QWK={res['mean_qwk']:.4f}")
    print("=" * 70)
    step_done(14, t0)

## Step 15 — OOF Ensemble & Threshold Optimization

1. Collect OOF predictions from all models
2. Average (ensemble)
3. Optimize thresholds on OOF
4. Final thresholds forced to [0.7, 1.5, 2.5, 3.5] (winner's choice)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 15 — OOF ENSEMBLE + THRESHOLD OPTIMIZATION
# ═══════════════════════════════════════════════════════════════
if is_done("threshold_opt"):
    step_skip(15, "THRESHOLD OPTIMIZATION")
else:
    t0 = step_header(15, "OOF ENSEMBLE + THRESHOLD OPTIMIZATION")

    # ── Collect OOF predictions ───────────────────────────────
    # Only for APTOS 2019 (fold >= 0)
    mask_2019 = df["fold"] >= 0
    df_2019 = df[mask_2019].reset_index(drop=True)
    n_val = len(df_2019)

    oof_ensemble = np.zeros(n_val, dtype=np.float32)
    n_models = 0

    for config in MODEL_CONFIGS:
        name = config["name"]
        for seed_val in config["seeds"]:
            fold_preds = np.zeros(n_val, dtype=np.float32)
            all_folds_ok = True

            for fold in range(N_FOLDS):
                tag = f"{name}_s{seed_val}_f{fold}"
                oof_file = ARTIFACT_DIR / f"{tag}_oof.npy"

                if oof_file.exists():
                    fold_mask = df_2019["fold"] == fold
                    oof_data = np.load(str(oof_file))
                    fold_preds[fold_mask.values] = oof_data[:fold_mask.sum()]
                else:
                    all_folds_ok = False
                    break

            if all_folds_ok:
                oof_ensemble += fold_preds
                n_models += 1

    if n_models > 0:
        oof_ensemble /= n_models
        print(f"  Ensembled {n_models} models")

        # Clip predictions
        oof_clipped = np.clip(oof_ensemble, 0, 4)
        true_labels = df_2019["label"].values

        # ── Optimize thresholds ───────────────────────────────
        def apply_thresholds(preds, thresholds):
            """Convert regression predictions to class labels using thresholds."""
            result = np.zeros_like(preds, dtype=int)
            for i, t in enumerate(thresholds):
                result[preds >= t] = i + 1
            return result

        def neg_qwk(thresholds, preds, labels):
            t = sorted(thresholds)
            pred_cls = apply_thresholds(preds, t)
            return -qwk(labels, pred_cls)

        # Scipy optimization
        from scipy.optimize import minimize
        initial = [0.5, 1.5, 2.5, 3.5]
        result = minimize(
            neg_qwk, initial, args=(oof_clipped, true_labels),
            method="Nelder-Mead",
            options={"maxiter": 5000}
        )
        opt_thresholds = sorted(result.x.tolist())
        opt_qwk = -result.fun

        print(f"  Optimized thresholds: {[f'{t:.3f}' for t in opt_thresholds]}")
        print(f"  Optimized QWK: {opt_qwk:.4f}")

        # ── Force winner's final thresholds ───────────────────
        FINAL_THRESHOLDS = [0.7, 1.5, 2.5, 3.5]
        final_preds = apply_thresholds(oof_clipped, FINAL_THRESHOLDS)
        final_qwk = qwk(true_labels, final_preds)

        print(f"\n  Winner thresholds: {FINAL_THRESHOLDS}")
        print(f"  Winner QWK: {final_qwk:.4f}")

        # Use whichever is better
        if opt_qwk > final_qwk:
            THRESHOLDS = opt_thresholds
            best_oof_qwk = opt_qwk
            print(f"\n  Using OPTIMIZED thresholds (better)")
        else:
            THRESHOLDS = FINAL_THRESHOLDS
            best_oof_qwk = final_qwk
            print(f"\n  Using WINNER thresholds (better)")

        # ── Metrics ───────────────────────────────────────────
        final_labels = apply_thresholds(oof_clipped, THRESHOLDS)
        acc = accuracy_score(true_labels, final_labels)
        f1 = f1_score(true_labels, final_labels, average="weighted")
        prec = precision_score(true_labels, final_labels, average="weighted")
        rec = recall_score(true_labels, final_labels, average="weighted")

        print(f"\n  OOF Metrics (ensemble):")
        print(f"    QWK:       {best_oof_qwk:.4f}")
        print(f"    Accuracy:  {acc*100:.2f}%")
        print(f"    F1:        {f1*100:.2f}%")
        print(f"    Precision: {prec*100:.2f}%")
        print(f"    Recall:    {rec*100:.2f}%")

        # Save
        save_json({
            "optimized_thresholds": opt_thresholds,
            "winner_thresholds": FINAL_THRESHOLDS,
            "selected_thresholds": THRESHOLDS,
            "oof_qwk": float(best_oof_qwk),
            "accuracy": float(acc),
            "f1": float(f1),
            "n_models": n_models,
        }, ARTIFACT_DIR / "threshold_results.json")

        state_save("oof_qwk", float(best_oof_qwk))
        state_save("oof_acc", float(acc))
        state_save("thresholds", THRESHOLDS)

        np.save(str(ARTIFACT_DIR / "oof_ensemble.npy"), oof_clipped)
        np.save(str(ARTIFACT_DIR / "oof_labels.npy"), true_labels)

    else:
        print("  ⚠️ No OOF predictions found. Run Step 14 first.")
        THRESHOLDS = [0.7, 1.5, 2.5, 3.5]

    mark_done("threshold_opt")
    step_done(15, t0)

## Step 16 — Stage 2: Pseudo-Labeling + Fine-tuning

**Winner Stage 2:**
1. Generate soft pseudo-labels for test data using Stage 1 models
2. Add IDRiD dataset (label = (gt + pred) / 2)
3. Add Messidor dataset (label = clip(pred, mean ± 0.5))
4. Fine-tune each Stage 1 model for 10 more epochs

**Critical:** Pseudo-labels used ONLY in training, NEVER in validation.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 16 — STAGE 2: PSEUDO-LABELING + FINE-TUNING
# Winner: add pseudo-labeled test + IDRiD + Messidor, train 10 more ep
# ═══════════════════════════════════════════════════════════════
if is_done("stage2_training"):
    step_skip(16, "STAGE 2 PSEUDO-LABELING")
else:
    t0 = step_header(16, "STAGE 2: PSEUDO-LABELING + FINE-TUNING")

    # ── Generate pseudo labels for APTOS 2019 test set ────────
    test_img_dir = APTOS19_DIR / "test_images"
    test_csv = APTOS19_DIR / "test.csv"

    pseudo_frames = []

    if test_csv.exists() and test_img_dir.exists():
        df_test = pd.read_csv(test_csv)
        df_test["image_path"] = df_test["id_code"].apply(
            lambda x: str(test_img_dir / f"{x}.png"))
        df_test = df_test[df_test["image_path"].apply(
            lambda p: Path(p).exists())].reset_index(drop=True)

        if len(df_test) > 0:
            print(f"  Generating pseudo-labels for {len(df_test)} test images...")

            # Use best fold of first available model for pseudo-labels
            pseudo_preds = np.zeros(len(df_test), dtype=np.float32)
            n_used = 0

            for config in MODEL_CONFIGS:
                name = config["name"]
                backbone = config["backbone"]
                for seed_val in config["seeds"]:
                    # Find best fold
                    for fold in range(N_FOLDS):
                        tag = f"{name}_s{seed_val}_f{fold}"
                        ckpt_path = CKPT_DIR / f"{tag}_best.pt"
                        if ckpt_path.exists():
                            ckpt = safe_load(ckpt_path, DEVICE)
                            model = build_model(backbone, pretrained=False)
                            model.load_state_dict(ckpt["model_state"])

                            # Apply EMA if available
                            if "ema" in ckpt:
                                ema = EMA(model, decay=0.9999)
                                ema.load_state_dict(ckpt["ema"])
                                ema.apply(model)

                            model.eval()
                            ds = DRDataset(df_test, transform=build_val_transforms(384), img_size=384)
                            dl = make_loader(ds, BS_MAP[384], shuffle=False)

                            preds = []
                            with torch.no_grad():
                                for imgs, _ in dl:
                                    out = model(imgs.to(DEVICE))
                                    preds.extend(out.cpu().numpy().tolist())

                            pseudo_preds += np.array(preds)
                            n_used += 1
                            del model
                            gc.collect()
                            if DEVICE == "cuda":
                                torch.cuda.empty_cache()
                            break  # One fold per model is enough

            if n_used > 0:
                pseudo_preds /= n_used
                pseudo_preds = np.clip(pseudo_preds, 0, 4)
                df_test["label"] = pseudo_preds  # SOFT labels
                df_test["source"] = "pseudo_test"
                pseudo_frames.append(df_test[["id_code", "label", "image_path", "source"]])
                print(f"  Pseudo-labeled {len(df_test)} test images using {n_used} models")
    else:
        print("  Test images not found — skipping pseudo-labels")

    # ── Build Stage 2 training set ────────────────────────────
    # Original training data (always fold=-1 for Stage 2, no validation on pseudo)
    df_s2 = df.copy()

    # Add pseudo-labeled test data
    if pseudo_frames:
        df_pseudo = pd.concat(pseudo_frames, ignore_index=True)
        df_pseudo["fold"] = -1  # NEVER in validation
        df_s2 = pd.concat([df_s2, df_pseudo], ignore_index=True)
        print(f"  Added {len(df_pseudo)} pseudo-labeled samples")

    # IDRiD handling (if available)
    if (IDRID_DIR / "labels.csv").exists():
        print("  IDRiD: found — applying label smoothing (gt + pred) / 2")
        # Implementation would go here when IDRiD data is available

    # Messidor handling (if available)
    if (MESSIDOR_DIR / "labels.csv").exists():
        print("  Messidor: found — applying bounded label adjustment")
        # Implementation would go here when Messidor data is available

    print(f"\n  Stage 2 dataset: {len(df_s2):,} total")

    # ── Fine-tune each model for 10 more epochs ──────────────
    S2_EPOCHS = 10
    S2_LR = 3e-5
    S2_SIZE = 384  # Use 384 for Stage 2 fine-tuning (memory-safe)
    S2_BS = BS_MAP[S2_SIZE]
    S2_ACCUM = GRAD_ACCUM_MAP[S2_SIZE]

    for config in MODEL_CONFIGS:
        backbone = config["backbone"]
        name = config["name"]

        for seed_val in config["seeds"]:
            for fold in range(N_FOLDS):
                tag = f"{name}_s{seed_val}_f{fold}"
                s2_tag = f"s2_{tag}"
                s1_ckpt = CKPT_DIR / f"{tag}_best.pt"
                s2_ckpt = CKPT_DIR / f"{s2_tag}_best.pt"

                if is_done(s2_tag) and s2_ckpt.exists():
                    print(f"  ✅ {s2_tag}: already done")
                    continue

                if not s1_ckpt.exists():
                    print(f"  ⚠️ {tag}: no Stage 1 checkpoint, skipping")
                    continue

                print(f"\n  ── Stage 2: {tag} ──")
                seed_everything(seed_val + fold + 1000)

                # Load Stage 1 model
                model = build_model(backbone, pretrained=False)
                ckpt = safe_load(s1_ckpt, DEVICE)
                model.load_state_dict(ckpt["model_state"])
                model.unfreeze_all()
                model.to(DEVICE)
                ema = EMA(model, decay=0.9999)

                criterion = nn.SmoothL1Loss()

                # Stage 2 training set: everything except validation fold
                df_s2_train = df_s2[df_s2["fold"] != fold].reset_index(drop=True)
                df_val = df[df["fold"] == fold].reset_index(drop=True)

                tr_ds = DRDataset(df_s2_train, transform=build_train_transforms(S2_SIZE), img_size=S2_SIZE)
                va_ds = DRDataset(df_val, transform=build_val_transforms(S2_SIZE), img_size=S2_SIZE)
                tr_dl = make_loader(tr_ds, S2_BS, shuffle=True, drop_last=True)
                va_dl = make_loader(va_ds, S2_BS, shuffle=False)

                opt = torch.optim.AdamW(model.parameters(), lr=S2_LR, weight_decay=WD)
                sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=S2_EPOCHS)
                scaler = torch.amp.GradScaler("cuda") if USE_AMP else None

                best_qwk = ckpt.get("val_qwk", 0)
                best_state = deepcopy(model.state_dict())

                for ep in range(S2_EPOCHS):
                    model.train()
                    epoch_loss = 0
                    opt.zero_grad()

                    for step, (imgs, labels) in enumerate(tr_dl):
                        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
                        if scaler:
                            with torch.amp.autocast("cuda"):
                                loss = criterion(model(imgs), labels) / S2_ACCUM
                            scaler.scale(loss).backward()
                            if (step + 1) % S2_ACCUM == 0:
                                scaler.unscale_(opt)
                                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                                scaler.step(opt)
                                scaler.update()
                                opt.zero_grad()
                        else:
                            loss = criterion(model(imgs), labels) / S2_ACCUM
                            loss.backward()
                            if (step + 1) % S2_ACCUM == 0:
                                torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                                opt.step()
                                opt.zero_grad()
                        epoch_loss += loss.item() * S2_ACCUM
                        ema.update(model)

                    sched.step()

                    # Validate
                    ema.apply(model)
                    model.eval()
                    vp, vl = [], []
                    with torch.no_grad():
                        for imgs, labels in va_dl:
                            out = model(imgs.to(DEVICE))
                            vp.extend(out.cpu().numpy().tolist())
                            vl.extend(labels.numpy().astype(int).tolist())
                    ema.restore(model)

                    vp_cls = np.clip(np.round(np.clip(vp, 0, 4)), 0, 4).astype(int)
                    v_qwk = qwk(vl, vp_cls)
                    tl = epoch_loss / max(len(tr_dl), 1)

                    flag = ""
                    if v_qwk > best_qwk:
                        best_qwk = v_qwk
                        best_state = deepcopy(model.state_dict())
                        flag = " ★"

                    print(f"    S2 Ep{ep+1:02d} Loss={tl:.4f} QWK={v_qwk:.4f}{flag}")

                # Save Stage 2 best
                torch.save({
                    "model_state": best_state,
                    "val_qwk": best_qwk,
                    "backbone": backbone,
                    "stage": 2,
                    "ema": ema.state_dict(),
                }, s2_ckpt)

                mark_done(s2_tag)
                print(f"  ✅ {s2_tag}: QWK={best_qwk:.4f}")

                del model, tr_ds, va_ds, tr_dl, va_dl, opt, sched
                gc.collect()
                if DEVICE == "cuda":
                    torch.cuda.empty_cache()

    mark_done("stage2_training")
    step_done(16, t0)

## Step 17 — Final Metrics & Confusion Matrix

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 17 — FINAL METRICS & VISUALIZATION
# ═══════════════════════════════════════════════════════════════
if is_done("final_metrics"):
    step_skip(17, "FINAL METRICS")
else:
    t0 = step_header(17, "FINAL METRICS & VISUALIZATION")

    thr_file = ARTIFACT_DIR / "threshold_results.json"
    if thr_file.exists():
        results = load_json(thr_file)
        THRESHOLDS = results.get("selected_thresholds", [0.7, 1.5, 2.5, 3.5])
    else:
        THRESHOLDS = [0.7, 1.5, 2.5, 3.5]

    oof_file = ARTIFACT_DIR / "oof_ensemble.npy"
    labels_file = ARTIFACT_DIR / "oof_labels.npy"

    if oof_file.exists() and labels_file.exists():
        oof_preds = np.load(str(oof_file))
        true_labels = np.load(str(labels_file))

        def apply_thresholds(preds, thresholds):
            result = np.zeros_like(preds, dtype=int)
            for i, t in enumerate(thresholds):
                result[preds >= t] = i + 1
            return result

        pred_labels = apply_thresholds(oof_preds, THRESHOLDS)

        # Full metrics
        metrics = {
            "qwk": float(qwk(true_labels, pred_labels)),
            "accuracy": float(accuracy_score(true_labels, pred_labels)),
            "f1": float(f1_score(true_labels, pred_labels, average="weighted")),
            "precision": float(precision_score(true_labels, pred_labels, average="weighted")),
            "recall": float(recall_score(true_labels, pred_labels, average="weighted")),
        }

        print("  Final OOF Metrics:")
        for k, v in metrics.items():
            print(f"    {k:12s}: {v:.4f}")

        # Classification report
        print(f"\n  Classification Report:")
        print(classification_report(
            true_labels, pred_labels,
            target_names=[GRADE_MAP[i] for i in range(5)]
        ))

        # Confusion matrix
        cm = confusion_matrix(true_labels, pred_labels)
        fig, ax = plt.subplots(1, 1, figsize=(8, 6))
        im = ax.imshow(cm, cmap="Blues")
        ax.set_xlabel("Predicted")
        ax.set_ylabel("True")
        ax.set_title(f"Confusion Matrix (QWK={metrics['qwk']:.4f})")
        ax.set_xticks(range(5))
        ax.set_yticks(range(5))
        ax.set_xticklabels([GRADE_MAP[i] for i in range(5)], rotation=45, ha="right")
        ax.set_yticklabels([GRADE_MAP[i] for i in range(5)])
        for i in range(5):
            for j in range(5):
                ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                        color="white" if cm[i, j] > cm.max() / 2 else "black")
        plt.tight_layout()
        plt.savefig(PLOT_DIR / "confusion_matrix.png", dpi=150)
        plt.close()
        print(f"  Saved: {PLOT_DIR / 'confusion_matrix.png'}")

        save_json(metrics, LOG_DIR / "final_metrics.json")
        state_save("final_qwk", metrics["qwk"])
    else:
        print("  ⚠️ OOF predictions not found. Run Steps 14-15 first.")

    mark_done("final_metrics")
    step_done(17, t0)

## Step 18 — Model Export

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 18 — MODEL EXPORT
# ═══════════════════════════════════════════════════════════════
if is_done("export"):
    step_skip(18, "MODEL EXPORT")
else:
    t0 = step_header(18, "MODEL EXPORT")
    EXPORT_DIR.mkdir(parents=True, exist_ok=True)

    # Export all best checkpoints
    exported = 0
    for config in MODEL_CONFIGS:
        name = config["name"]
        for seed_val in config["seeds"]:
            for fold in range(N_FOLDS):
                # Prefer Stage 2 if available
                s2_tag = f"s2_{name}_s{seed_val}_f{fold}"
                s1_tag = f"{name}_s{seed_val}_f{fold}"

                s2_path = CKPT_DIR / f"{s2_tag}_best.pt"
                s1_path = CKPT_DIR / f"{s1_tag}_best.pt"

                src = s2_path if s2_path.exists() else s1_path
                if src.exists():
                    dst = EXPORT_DIR / f"{s1_tag}.pt"
                    shutil.copy2(src, dst)
                    exported += 1

    # Save threshold and config
    save_json({
        "thresholds": state_get("thresholds", [0.7, 1.5, 2.5, 3.5]),
        "models": [c["name"] for c in MODEL_CONFIGS],
        "img_size": 384,
        "grade_map": {str(k): v for k, v in GRADE_MAP.items()},
    }, EXPORT_DIR / "config.json")

    print(f"  Exported {exported} model checkpoints")
    print(f"  Path: {EXPORT_DIR}")

    mark_done("export")
    step_done(18, t0)

## Step 19 — Inference Pipeline

Complete inference: preprocess → TTA → ensemble average → clip [0,4] → apply thresholds

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 19 — INFERENCE PIPELINE
# preprocess → TTA → ensemble average → clip → thresholds
# ═══════════════════════════════════════════════════════════════
t0 = step_header(19, "INFERENCE PIPELINE")

THRESHOLDS = state_get("thresholds", [0.7, 1.5, 2.5, 3.5])

def apply_thresholds(preds, thresholds):
    """Convert regression predictions to DR grades."""
    result = np.zeros_like(preds, dtype=int)
    for i, t in enumerate(thresholds):
        result[preds >= t] = i + 1
    return result

@torch.no_grad()
def predict_single(image_rgb, models, device=DEVICE, img_size=384):
    """
    Full inference pipeline for a single image.
    image_rgb: numpy array (H, W, 3) in RGB
    models: list of (model, backbone_name) tuples
    Returns: (grade, raw_score, all_scores)
    """
    # Step 1: Minimal preprocessing
    img = cv2.resize(image_rgb, (img_size, img_size), interpolation=cv2.INTER_AREA)

    # Step 2: TTA transforms
    tta_tfs = build_tta_transforms(img_size)

    # Step 3: Ensemble over all models × all TTA views
    all_preds = []
    for model, _ in models:
        model.eval()
        for ttf in tta_tfs:
            inp = ttf(image=img)["image"].unsqueeze(0).to(device)
            out = model(inp).item()
            all_preds.append(out)

    # Step 4: Average predictions
    mean_pred = np.mean(all_preds)

    # Step 5: Clip to [0, 4]
    clipped = np.clip(mean_pred, 0, 4)

    # Step 6: Apply thresholds
    grade = apply_thresholds(np.array([clipped]), THRESHOLDS)[0]

    return grade, clipped, all_preds


print(f"  Thresholds: {THRESHOLDS}")
print(f"  Pipeline: preprocess → TTA(5) → ensemble → clip → threshold")
print(f"  Ready for inference")

step_done(19, t0)

## Step 20 — Deployment (Streamlit App)

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 20 — STREAMLIT DEPLOYMENT
# ═══════════════════════════════════════════════════════════════
if is_done("deployment"):
    step_skip(20, "DEPLOYMENT")
else:
    t0 = step_header(20, "DEPLOYMENT — STREAMLIT APP")
    DEPLOY_DIR.mkdir(parents=True, exist_ok=True)

    # Copy model files
    for f in EXPORT_DIR.glob("*"):
        if f.is_file():
            shutil.copy2(f, DEPLOY_DIR / f.name)

    # ── app.py ────────────────────────────────────────────────
    app_lines = [
        "import os, json, numpy as np, cv2, torch, torch.nn as nn",
        "import torch.nn.functional as F, timm, streamlit as st",
        "import albumentations as A",
        "from albumentations.pytorch import ToTensorV2",
        "from PIL import Image",
        "from torch.nn.parameter import Parameter",
        "",
        "IMAGENET_MEAN = [0.485, 0.456, 0.406]",
        "IMAGENET_STD = [0.229, 0.224, 0.225]",
        'GRADE_MAP = {0: "No DR", 1: "Mild", 2: "Moderate", 3: "Severe", 4: "Proliferative"}',
        "IMG_SIZE = 384",
        'DEVICE = "cuda" if torch.cuda.is_available() else "cpu"',
        "",
        "class GeM(nn.Module):",
        "    def __init__(self, p=3, eps=1e-6):",
        "        super().__init__()",
        "        self.p = Parameter(torch.ones(1) * p)",
        "        self.eps = eps",
        "    def forward(self, x):",
        "        return F.avg_pool2d(x.clamp(min=self.eps).pow(self.p),",
        "                           (x.size(-2), x.size(-1))).pow(1./self.p)",
        "",
        "class DRModel(nn.Module):",
        "    def __init__(self, backbone_name, pretrained=False, drop=0.5):",
        "        super().__init__()",
        '        self.backbone = timm.create_model(backbone_name, pretrained=pretrained, num_classes=0, global_pool="")',
        "        fd = self.backbone.num_features",
        "        self.pool = GeM()",
        "        self.head = nn.Sequential(nn.Flatten(), nn.Linear(fd, 256), nn.BatchNorm1d(256),",
        "            nn.ReLU(True), nn.Dropout(drop), nn.Linear(256, 1))",
        "    def forward(self, x):",
        "        return self.head(self.pool(self.backbone(x))).squeeze(-1)",
        "",
        "def is_retinal_image(img_rgb, min_size=100):",
        "    h, w = img_rgb.shape[:2]",
        "    if h < min_size or w < min_size: return False, 'Too small'",
        "    gray = cv2.cvtColor(img_rgb, cv2.COLOR_RGB2GRAY)",
        "    if float(gray.mean()) < 5: return False, 'Completely black'",
        "    if float(gray.mean()) > 250: return False, 'Completely white'",
        "    return True, 'Valid'",
        "",
        'st.set_page_config(page_title="DR Grading", page_icon="\\U0001FA7A", layout="wide")',
        'st.title("\\U0001FA7A Diabetic Retinopathy Grading")',
        'st.warning("RESEARCH USE ONLY")',
        "",
        'uploaded = st.file_uploader("Upload fundus image", type=["jpg", "jpeg", "png"])',
        "if uploaded:",
        '    pil = Image.open(uploaded).convert("RGB")',
        "    img = np.array(pil)",
        "    valid, reason = is_retinal_image(img)",
        "    if not valid:",
        '        st.error(f"Invalid: {reason}")',
        "        st.stop()",
        '    st.image(pil, caption="Uploaded Image", width=400)',
        '    st.info("Model inference would run here with loaded checkpoints.")',
    ]
    (DEPLOY_DIR / "app.py").write_text("\n".join(app_lines))
    print("  Created: app.py")

    # ── requirements.txt ──────────────────────────────────────
    (DEPLOY_DIR / "requirements.txt").write_text(
        "torch>=2.1\ntorchvision\ntimm>=1.0.0\nalbumentations>=1.4.0\n"
        "opencv-python-headless\nstreamlit>=1.35.0\npillow<11.0\nnumpy\n"
    )
    print("  Created: requirements.txt")

    # ── README.md ─────────────────────────────────────────────
    (DEPLOY_DIR / "README.md").write_text(
        "# DR Grading v21\n\n"
        "## Run\n```\nstreamlit run app.py\n```\n\n"
        "## Models\n8-model ensemble (4 backbones × 2 seeds)\n"
        "SmoothL1Loss regression + GeM pooling + threshold tuning\n"
    )
    print("  Created: README.md")

    mark_done("deployment")
    step_done(20, t0)

## Step 21 — Final Pipeline Summary

In [ ]:
# ═══════════════════════════════════════════════════════════════
# STEP 21 — FINAL PIPELINE SUMMARY
# ═══════════════════════════════════════════════════════════════
print("\n" + "=" * 70)
print("  DIABETIC RETINOPATHY GRADING — v21 FINAL SUMMARY")
print("=" * 70)

# Device
info = load_json(LOG_DIR / "system_info.json") if (LOG_DIR / "system_info.json").exists() else {}
print(f"  Device:     {info.get('device', 'unknown').upper()}")
if info.get("gpu_name"):
    print(f"  GPU:        {info['gpu_name']}")
print(f"  PyTorch:    {info.get('pytorch', 'unknown')}")

# Dataset
if (ARTIFACT_DIR / "df_kfold.parquet").exists():
    df_final = pd.read_parquet(ARTIFACT_DIR / "df_kfold.parquet")
    print(f"  Dataset:    {len(df_final):,} images")
    for src, cnt in df_final["source"].value_counts().items():
        print(f"    {src}: {cnt:,}")

# Models
print(f"\n  Architecture: 8-model ensemble")
print(f"    Backbones: EfficientNet-B4, EfficientNetV2-B1, EfficientNet-B3, SE-ResNeXt50")
print(f"    Pooling:   GeM (learnable)")
print(f"    Loss:      SmoothL1Loss (regression)")
print(f"    EMA:       decay=0.9999")

# Metrics
if (LOG_DIR / "final_metrics.json").exists():
    m = load_json(LOG_DIR / "final_metrics.json")
    print(f"\n  Metrics:")
    print(f"    QWK:       {m.get('qwk', 'N/A')}")
    print(f"    Accuracy:  {m.get('accuracy', 'N/A')}")
    print(f"    F1:        {m.get('f1', 'N/A')}")
    print(f"    Precision: {m.get('precision', 'N/A')}")
    print(f"    Recall:    {m.get('recall', 'N/A')}")

# Thresholds
thresholds = state_get("thresholds", [0.7, 1.5, 2.5, 3.5])
print(f"\n  Thresholds: {thresholds}")

# Completed steps
done = sorted(FLAG_DIR.glob("*.done"))
print(f"\n  ✅ Steps completed: {len(done)}")

# Paths
print(f"\n  Artifacts: {ARTIFACT_DIR}")
print(f"  Export:    {EXPORT_DIR}")
print(f"  Deploy:    {DEPLOY_DIR}")

print("\n" + "=" * 70)
print("  ⚠️ RESEARCH USE ONLY — NOT FOR CLINICAL DEPLOYMENT")
print("=" * 70)

## Utility — Reset Pipeline (optional)

**Only run this if you want to restart from scratch.**
Clears all flags, cached artifacts, and training state.

In [ ]:
# ═══════════════════════════════════════════════════════════════
# UTILITY — RESET PIPELINE (uncomment to use)
# ═══════════════════════════════════════════════════════════════

# WARNING: This will delete ALL progress!
# Uncomment the lines below to reset.

# for f in FLAG_DIR.glob("*.done"):
#     f.unlink()
#     print(f"  Cleared: {f.name}")
#
# for f in ARTIFACT_DIR.glob("*"):
#     if f.is_file():
#         f.unlink()
#
# if STATE_FILE.exists():
#     STATE_FILE.unlink()
#
# print("  ✅ Pipeline reset. Re-run all cells from Step 0.")